# Phase 4C.1C.0D.1: Colab T4 Resumable 15-Run Training Notebook V2

This notebook provides a clean, zero-credential execution harness for the 15-run Stage 1 learning curve matrix on Google Colab NVIDIA T4.

### Invariants & Guarantees
* **Zero Google Drive dependency**: Everything runs hermetically in `/content/`.
* **Zero credentials / GitHub tokens**: Zero repository cloning, zero personal access tokens.
* **Zero PyTorch reinstallation**: Inherits the Colab runtime PyTorch CUDA stack.
* **Sealed Artifact Delegation**: All 15 runs are managed by `phase_4c1_t4_execute_all_stage1.sh`.
* **Three Sample Sizes**: N=50, N=100, N=250.
* **Five Seeds**: 42, 1337, 2025, 3407, 9001.
* **Inner Validation Cohort**: Exactly 91 sources (182 paired samples), 0 development overlap, 0 locked-test access.
* **Stage 2 Invocations**: Exactly 0 (Stage 1 frozen backbone only).

### Required Uploads to `/content/forensics-transfer/`
1. `phase_4c1_code_79bb115.tar.gz` (SHA-256: `5e775a7708d1555bf22f56dceb5358e26e07dd224c4b6186f575aff4d2b590d5`)
2. `phase_4c1_binary_n250_reusable.tar` (SHA-256: `d49a106f0c4991ca8d79776277cbf7331df209157725c438288720dc42226a27`)
3. `phase_4c1_t4_execute_all_stage1.sh` (SHA-256: `deb3f04dd7c1039c4f3248f98a27f4eaf8e11f206a218125e949e5869da8ce05`)


In [ ]:
# Hardware and Environment Preflight
import sys
import torch
import torchvision

print("=" * 60)
print("PHASE 4C.1 COLAB T4 PREFLIGHT (NOTEBOOK V2)")
print("=" * 60)
print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {sys.platform}")
print(f"PyTorch: {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

assert sys.platform.startswith("linux"), f"Must run on Linux Colab, got {sys.platform}"
assert torch.cuda.is_available(), "CUDA is not available. Please connect to a GPU runtime."

gpu_name = torch.cuda.get_device_name(0)
print(f"GPU Model: {gpu_name}")
assert "T4" in gpu_name.upper(), f"NVIDIA T4 GPU required, got {gpu_name}"

x = torch.ones((16, 16), device="cuda")
assert x.sum().item() == 256.0, "CUDA tensor sum sanity check failed"
print("[+] GPU Preflight Passed: NVIDIA T4 is active and ready.")


In [ ]:
# Transfer Artifacts Integrity and Checksum Verification (Fail-Closed)
import hashlib
from pathlib import Path

TRANSFER_DIR = Path("/content/forensics-transfer")

EXPECTED_ARTIFACTS = {
    "code_archive": {
        "filename": "phase_4c1_code_79bb115.tar.gz",
        "bytes": 758350,
        "sha256": "5e775a7708d1555bf22f56dceb5358e26e07dd224c4b6186f575aff4d2b590d5"
    },
    "reusable_bundle": {
        "filename": "phase_4c1_binary_n250_reusable.tar",
        "bytes": 724633600,
        "sha256": "d49a106f0c4991ca8d79776277cbf7331df209157725c438288720dc42226a27"
    },
    "operator_script": {
        "filename": "phase_4c1_t4_execute_all_stage1.sh",
        "bytes": 20005,
        "sha256": "deb3f04dd7c1039c4f3248f98a27f4eaf8e11f206a218125e949e5869da8ce05"
    }
}

print(f"Verifying transfer artifacts in {TRANSFER_DIR}...")
assert TRANSFER_DIR.exists(), f"Transfer directory missing: {TRANSFER_DIR}. Please upload files first."

for key, spec in EXPECTED_ARTIFACTS.items():
    fn = spec["filename"]
    exp_bytes = spec["bytes"]
    exp_sha = spec["sha256"]
    fp = TRANSFER_DIR / fn
    assert fp.exists(), f"MISSING ARTIFACT: {fn} not found in {TRANSFER_DIR}. Upload required."
    act_bytes = fp.stat().st_size
    assert act_bytes == exp_bytes, f"BYTE MISMATCH for {fn}: expected {exp_bytes}, got {act_bytes}."
    with open(fp, "rb") as f:
        act_sha = hashlib.sha256(f.read()).hexdigest().lower()
    assert act_sha == exp_sha.lower(), f"SHA-256 MISMATCH for {fn}: expected {exp_sha}, got {act_sha}."
    print(f"[+] Verified {key}: {fn} ({act_bytes} bytes, SHA-256 matches)")

print("[+] All 3 transfer artifacts are valid, uncorrupted, and sealed.")


### Execution Gate

By default, `EXECUTE = False` to prevent accidental training runs. Set `EXECUTE = True` only when ready to launch the 15-run matrix.


In [ ]:
# Execution Gate Flag
EXECUTE = False
print(f"EXECUTE is currently set to: {EXECUTE}")
if not EXECUTE:
    print("[SAFETY NOTICE] Execution is disabled by default. Set EXECUTE = True to run training.")


In [ ]:
# Launch Resumable 15-Run Operator Script
import subprocess
import sys

if not EXECUTE:
    print("[DRY-RUN] EXECUTE is False. Operator script invocation skipped.")
    print("[DRY-RUN] No training runs initiated. Training runs = 0.")
else:
    cmd = ["bash", "/content/forensics-transfer/phase_4c1_t4_execute_all_stage1.sh"]
    print(f"[*] Executing operator: {' '.join(cmd)}")
    proc = subprocess.run(cmd)
    if proc.returncode != 0:
        print(f"[-] Operator failed with exit code {proc.returncode}")
        sys.exit(proc.returncode)
    else:
        print("[+] Operator execution completed successfully.")


In [ ]:
# Read-Only Output Integrity Audit
import json
from pathlib import Path

OUTPUT_ROOT = Path("/content/phase_4c1_outputs")
DOWNLOAD_DIR = OUTPUT_ROOT / "download"

if not EXECUTE:
    print("[DRY-RUN] Skipping output audit since EXECUTE is False.")
else:
    print(f"Auditing outputs in {OUTPUT_ROOT}...")
    assert OUTPUT_ROOT.exists(), f"Output root not found: {OUTPUT_ROOT}"
    lock_file = OUTPUT_ROOT / "phase4c1_environment_lock.json"
    assert lock_file.exists(), "Environment lock file missing from outputs"
    
    SAMPLE_SIZES = [50, 100, 250]
    SEEDS = [42, 1337, 2025, 3407, 9001]
    REQUIRED_ARTIFACTS = [
        "best_checkpoint.pt",
        "run_receipt.json",
        "epoch_history.json",
        "predictions.json",
        "training_history.csv",
        "metrics.json",
        "environment.json",
        "environment-binding.json",
        "checksums.json"
    ]
    
    for size in SAMPLE_SIZES:
        for seed in SEEDS:
            run_dir = OUTPUT_ROOT / f"n{size}_seed_{seed}"
            assert run_dir.exists(), f"Run directory missing: {run_dir}"
            for art in REQUIRED_ARTIFACTS:
                fp = run_dir / art
                assert fp.exists(), f"Artifact {art} missing in {run_dir}"
                assert fp.stat().st_size > 0, f"Artifact {art} in {run_dir} is empty"
            with open(run_dir / "run_receipt.json", "r", encoding="utf-8") as rf:
                rcpt = json.load(rf)
            assert rcpt.get("status") == "completed", f"Run {run_dir} not completed"
            assert rcpt.get("locked_test_access", 0) == 0, f"Locked test access detected in {run_dir}"
            assert rcpt.get("stage_2_invocation", 0) == 0, f"Stage 2 invocation detected in {run_dir}"
    
    for sz in SAMPLE_SIZES:
        grp_tar = DOWNLOAD_DIR / f"n{sz}_results.tar.gz"
        assert grp_tar.exists(), f"Group archive missing: {grp_tar}"
    final_tar = DOWNLOAD_DIR / "phase_4c1_all_15_runs_results.tar.gz"
    assert final_tar.exists(), f"Final archive missing: {final_tar}"
    print("[+] All 15 runs and archives verified successfully. Zero leakage verified.")
